# Notebook 4: Advanced Prompt Engineering Techniques
### ReAct, Self-Consistency, Prompt Chaining, and RAG — fully simulated

**No external models needed.** All patterns implemented with pure Python.
The architecture is real — swap MockLLM for actual API calls.

---
Topics covered:
1. ReAct pattern (Reason + Act loop)
2. Self-consistency (majority voting)
3. Prompt chaining (multi-stage pipelines)
4. Retrieval-Augmented Generation (RAG) from scratch
5. Meta-prompting
6. Prompt compression strategies

In [ ]:
import json
import re
import math
import random
import numpy as np
from collections import Counter
from textwrap import dedent
print("Imports OK ✓")

## 1. ReAct Pattern — Reason + Act Loop
The foundational pattern behind AI agents. The model alternates between Thought (reasoning) and Action (tool call).

In [ ]:
# Simulate tool functions that a ReAct agent would call
# In production these would be real API calls to banking systems

class BankingToolbox:
    """Simulated banking tools for ReAct demo"""
    
    # Mock data
    _bureau_db = {
        "ABCDE1234F": {"cibil_score": 745, "active_loans": 1, "dpd_count_12m": 0,
                       "income_verified": 80000, "employer": "TCS", "years_employed": 5},
        "FGHIJ5678K": {"cibil_score": 680, "active_loans": 3, "dpd_count_12m": 2,
                       "income_verified": 55000, "employer": "Self", "years_employed": 2},
        "KLMNO9012P": {"cibil_score": 720, "active_loans": 2, "dpd_count_12m": 0,
                       "income_verified": 120000, "employer": "Infosys", "years_employed": 8},
    }
    
    _existing_loans = {
        "ABCDE1234F": [{"type": "Car Loan", "emi": 8500, "outstanding": 240000}],
        "FGHIJ5678K": [{"type": "Personal Loan", "emi": 12000, "outstanding": 450000},
                       {"type": "Credit Card", "emi": 5000, "outstanding": 80000},
                       {"type": "Home Loan", "emi": 22000, "outstanding": 3200000}],
        "KLMNO9012P": [{"type": "Home Loan", "emi": 35000, "outstanding": 4500000},
                       {"type": "Car Loan", "emi": 15000, "outstanding": 620000}],
    }
    
    @classmethod
    def call_bureau_api(cls, pan: str) -> dict:
        print(f"    [TOOL] call_bureau_api(pan='{pan}')")
        data = cls._bureau_db.get(pan)
        if not data:
            return {"error": "PAN not found in bureau"}
        return data
    
    @classmethod
    def get_existing_loans(cls, pan: str) -> list:
        print(f"    [TOOL] get_existing_loans(pan='{pan}')")
        return cls._existing_loans.get(pan, [])
    
    @classmethod
    def calculate_foir(cls, monthly_income: float, existing_emis: list, proposed_emi: float) -> dict:
        print(f"    [TOOL] calculate_foir(income={monthly_income}, existing={existing_emis}, proposed={proposed_emi})")
        total_obligations = sum(existing_emis) + proposed_emi
        foir = (total_obligations / monthly_income) * 100
        return {
            "total_obligations": total_obligations,
            "foir_pct": round(foir, 2),
            "policy_max": 55,
            "passes": foir <= 55
        }
    
    @classmethod
    def calculate_emi(cls, principal: float, annual_rate: float, tenure_months: int) -> float:
        print(f"    [TOOL] calculate_emi(P={principal}, rate={annual_rate}%, n={tenure_months}m)")
        r = annual_rate / (12 * 100)
        emi = principal * r * (1 + r)**tenure_months / ((1 + r)**tenure_months - 1)
        return round(emi, 2)

print("BankingToolbox ready ✓")

In [ ]:
# Simulate the ReAct loop
# Each step: Thought → Action → Observation → next Thought

def react_loan_eligibility(pan: str, requested_loan: float, rate: float, tenure_months: int):
    """Simulated ReAct agent for loan eligibility check"""
    
    print(f"\n{'='*65}")
    print(f"REACT AGENT: Loan Eligibility for PAN {pan}")
    print(f"Request: ₹{requested_loan:,.0f} @ {rate}% for {tenure_months} months")
    print(f"{'='*65}\n")
    
    # Step 1
    print("THOUGHT 1: I need to fetch bureau data to check CIBIL score and income.")
    bureau = BankingToolbox.call_bureau_api(pan)
    if 'error' in bureau:
        print(f"OBSERVATION 1: ERROR — {bureau['error']}")
        print("THOUGHT 2: Cannot proceed without bureau data. DECLINE: Insufficient data.")
        return {"decision": "Decline", "reason": "Bureau data unavailable"}
    print(f"OBSERVATION 1: CIBIL={bureau['cibil_score']}, Income=₹{bureau['income_verified']:,}, DPD_12M={bureau['dpd_count_12m']}")
    
    # Step 2: CIBIL check
    print(f"\nTHOUGHT 2: CIBIL is {bureau['cibil_score']}. Policy min is 700. ", end="")
    if bureau['cibil_score'] < 700:
        print("BELOW threshold.")
        print(f"OBSERVATION 2: CIBIL check FAILED ({bureau['cibil_score']} < 700)")
        print("THOUGHT 3: CIBIL failure is a hard stop. No need to check further.")
        return {"decision": "Decline", "reason": f"CIBIL {bureau['cibil_score']} below minimum 700"}
    print("ABOVE threshold. Continue to FOIR check.")
    
    # Step 3: Get existing loans
    print(f"\nTHOUGHT 3: Need existing EMI obligations to calculate FOIR.")
    loans = BankingToolbox.get_existing_loans(pan)
    existing_emis = [l['emi'] for l in loans]
    print(f"OBSERVATION 3: {len(loans)} existing loan(s). EMIs: {existing_emis}. Total: ₹{sum(existing_emis):,}")
    
    # Step 4: Calculate proposed EMI
    print(f"\nTHOUGHT 4: Calculate proposed EMI for the requested loan.")
    proposed_emi = BankingToolbox.calculate_emi(requested_loan, rate, tenure_months)
    print(f"OBSERVATION 4: Proposed EMI = ₹{proposed_emi:,}")
    
    # Step 5: FOIR
    print(f"\nTHOUGHT 5: Now calculate FOIR with all obligations.")
    foir_result = BankingToolbox.calculate_foir(bureau['income_verified'], existing_emis, proposed_emi)
    print(f"OBSERVATION 5: FOIR={foir_result['foir_pct']}% (Policy max: {foir_result['policy_max']}%) — {'PASS ✓' if foir_result['passes'] else 'FAIL ✗'}")
    
    # Final decision
    print(f"\nTHOUGHT 6: All checks complete. Synthesizing decision.")
    if foir_result['passes']:
        decision = "Approve"
        reason = f"CIBIL {bureau['cibil_score']} ✓, FOIR {foir_result['foir_pct']}% ≤ 55% ✓"
    else:
        decision = "Conditional Decline"
        excess = foir_result['foir_pct'] - 55
        reason = f"FOIR {foir_result['foir_pct']}% exceeds 55% by {excess:.1f}%"
    
    result = {"decision": decision, "reason": reason, "foir": foir_result, "cibil": bureau['cibil_score']}
    print(f"\n{'='*65}")
    print(f"FINAL ANSWER: {decision}")
    print(f"Reason: {reason}")
    print(f"{'='*65}")
    return result

# Test with two applicants
result1 = react_loan_eligibility("ABCDE1234F", 2_000_000, 8.5, 240)
print()
result2 = react_loan_eligibility("FGHIJ5678K", 1_000_000, 9.0, 180)

## 2. Self-Consistency — Majority Voting for High-Stakes Decisions

In [ ]:
# Simulate running the same prompt N times at temperature > 0
# Real LLMs have variance — self-consistency reduces it

def simulate_stochastic_llm_runs(base_decision, n_runs=5, error_rate=0.2):
    """
    Simulate N runs of an LLM at temperature=0.7.
    Some runs may produce different decisions due to stochasticity.
    """
    decisions = []
    alternatives = ["Approve", "Decline", "Refer to Human", "Conditional Approve"]
    alternatives = [d for d in alternatives if d != base_decision]
    
    for i in range(n_runs):
        if random.random() < error_rate:
            # Stochastic error — different decision
            decision = random.choice(alternatives)
        else:
            decision = base_decision
        reasoning_quality = random.uniform(0.6, 0.99)
        decisions.append({"run": i+1, "decision": decision, "confidence": round(reasoning_quality, 3)})
    
    return decisions

def self_consistency_vote(runs):
    """Apply majority voting and compute agreement score"""
    counts = Counter(r['decision'] for r in runs)
    winner = counts.most_common(1)[0]
    agreement = winner[1] / len(runs)
    return {
        "final_decision": winner[0],
        "vote_count": winner[1],
        "total_runs": len(runs),
        "agreement_pct": round(agreement * 100, 1),
        "vote_distribution": dict(counts),
        "route_to_human": agreement < 0.7  # Route if <70% agreement
    }

random.seed(42)

# Scenario 1: Clear case — high agreement expected
print("=== SCENARIO 1: Clear Approval (CIBIL 745, FOIR 38%) ===")
runs_clear = simulate_stochastic_llm_runs("Approve", n_runs=7, error_rate=0.05)
for r in runs_clear:
    print(f"  Run {r['run']}: {r['decision']:<25} (confidence={r['confidence']})")
vote1 = self_consistency_vote(runs_clear)
print(f"\n  Majority Vote: {vote1['final_decision']} ({vote1['vote_count']}/{vote1['total_runs']} = {vote1['agreement_pct']}%)")
print(f"  Route to Human: {vote1['route_to_human']}")

print()

# Scenario 2: Borderline case — low agreement → human review
print("=== SCENARIO 2: Borderline Case (CIBIL 702, FOIR 53.8%) ===")
runs_border = simulate_stochastic_llm_runs("Conditional Approve", n_runs=7, error_rate=0.40)
for r in runs_border:
    print(f"  Run {r['run']}: {r['decision']:<25} (confidence={r['confidence']})")
vote2 = self_consistency_vote(runs_border)
print(f"\n  Majority Vote: {vote2['final_decision']} ({vote2['vote_count']}/{vote2['total_runs']} = {vote2['agreement_pct']}%)")
print(f"  Route to Human: {vote2['route_to_human']}")
if vote2['route_to_human']:
    print("  → UNCERTAINTY DETECTED: Routing to human underwriter")
    print("  → This 'uncertainty detection' is MORE valuable than the majority vote itself!")

In [ ]:
# Cost-benefit analysis of self-consistency
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

loan_values = [100_000, 500_000, 1_000_000, 5_000_000, 10_000_000, 50_000_000]
n_runs_options = [1, 3, 5, 7]
cost_per_call = 0.05  # $0.05 per LLM call for a medium-length prompt
error_rate_single = 0.15  # 15% error rate with single run

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cost of consistency vs loan value
for n in n_runs_options:
    consistency_costs = [n * cost_per_call for _ in loan_values]
    potential_error_cost = [lv * error_rate_single * (1 - 0.02*n) for lv in loan_values]  # Error reduces with more runs
    axes[0].plot([lv/1e6 for lv in loan_values], consistency_costs, 
                label=f'N={n} runs (cost=${n*cost_per_call:.2f})', marker='o')

axes[0].set_xlabel('Loan Value (₹ millions)')
axes[0].set_ylabel('LLM Cost per Decision ($)')
axes[0].set_title('Self-Consistency API Cost\nvs Loan Value')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Agreement rate vs accuracy improvement
agreement_rates = [0.43, 0.57, 0.71, 0.86, 1.0]
accuracy_boost = [1.0, 1.05, 1.12, 1.18, 1.22]  # Relative accuracy gain
axes[1].bar([f'{r:.0%}' for r in agreement_rates], accuracy_boost, 
            color=['red', 'orange', 'yellow', 'lightgreen', 'green'], alpha=0.8, edgecolor='black')
axes[1].axhline(y=1.0, color='black', linestyle='--', label='Baseline (N=1)')
axes[1].axvline(x=1.5, color='red', linestyle='--', alpha=0.5, label='Route-to-human threshold')
axes[1].set_xlabel('Majority Agreement Rate')
axes[1].set_ylabel('Relative Accuracy')
axes[1].set_title('Accuracy Gain from Self-Consistency\nvs Agreement Level')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/self_consistency_analysis.png', dpi=150, bbox_inches='tight')
plt.close()
print("Self-consistency analysis saved.")

print("\nRule of thumb: For ₹50L+ credit decisions, N=5 self-consistency adds ≈$0.25")
print("That's negligible vs the cost of a single wrong credit decision.")

## 3. Prompt Chaining — KYC Document Pipeline

In [ ]:
# 3-stage KYC pipeline from the deck, fully simulated

# Stage 1: Extraction
def stage1_extract(raw_ocr_text: str) -> dict:
    """Stage 1: Extract structured fields from OCR text"""
    print("\n[STAGE 1: EXTRACTION]")
    print(f"Input: {len(raw_ocr_text)} chars of OCR text")
    
    # In production: call LLM with extraction prompt
    # Simulating extraction output
    extracted = {
        "pan_name": "RAJESH KUMAR",
        "aadhaar_name": "Rajesh Kumar",
        "pan_dob": "15/03/1985",
        "aadhaar_dob": "15/03/1985",
        "pan_number": "ABCDE1234F",
        "aadhaar_last4": "5678",
        "address": "42, MG Road, Bangalore - 560001",
        "mobile": None,  # Not found in documents
        "extraction_confidence": "High"
    }
    print(f"Output: {json.dumps(extracted, indent=2)}")
    return extracted

# Stage 2: Validation
def stage2_validate(extracted: dict) -> dict:
    """Stage 2: Cross-validate fields and check PMLA completeness"""
    print("\n[STAGE 2: VALIDATION]")
    
    validation = {}
    
    # Name consistency check
    if extracted['pan_name'] and extracted['aadhaar_name']:
        names_match = extracted['pan_name'].upper() == extracted['aadhaar_name'].upper()
        validation['name_match'] = {'pass': names_match, 
                                     'detail': f"PAN: '{extracted['pan_name']}' | Aadhaar: '{extracted['aadhaar_name']}'",
                                     'severity': 'OK' if names_match else 'HIGH'}
    
    # DOB consistency
    dob_match = extracted['pan_dob'] == extracted['aadhaar_dob']
    validation['dob_match'] = {'pass': dob_match,
                               'detail': f"PAN: {extracted['pan_dob']} | Aadhaar: {extracted['aadhaar_dob']}",
                               'severity': 'OK' if dob_match else 'CRITICAL'}
    
    # PAN format
    pan_valid = bool(re.match(r'^[A-Z]{5}[0-9]{4}[A-Z]$', extracted.get('pan_number', '') or ''))
    validation['pan_format'] = {'pass': pan_valid, 'detail': extracted.get('pan_number'), 'severity': 'OK' if pan_valid else 'CRITICAL'}
    
    # PMLA mandatory fields
    pmla_required = ['pan_name', 'pan_number', 'aadhaar_last4', 'address', 'pan_dob']
    missing = [f for f in pmla_required if not extracted.get(f)]
    validation['pmla_completeness'] = {'pass': len(missing) == 0,
                                        'missing': missing,
                                        'severity': 'OK' if not missing else 'HIGH'}
    
    validation['optional_missing'] = [f for f in ['mobile'] if not extracted.get(f)]
    validation['overall_pass'] = all(v['pass'] for k, v in validation.items() if isinstance(v, dict) and 'pass' in v)
    
    print(f"Output: {json.dumps(validation, indent=2)}")
    return validation

# Stage 3: Narrative
def stage3_narrative(extracted: dict, validation: dict) -> str:
    """Stage 3: Human-readable summary for onboarding officer"""
    print("\n[STAGE 3: NARRATIVE GENERATION]")
    
    status = "CLEAR FOR ONBOARDING" if validation['overall_pass'] else "REQUIRES REVIEW"
    
    narrative = f"""KYC VERIFICATION SUMMARY
{'='*50}
Customer: {extracted['pan_name']}
PAN: {extracted['pan_number']} | Aadhaar Last 4: {extracted['aadhaar_last4']}
DOB: {extracted['pan_dob']} | Address: {extracted['address']}

VERIFICATION STATUS: {status}

Checks Performed:
  Name consistency (PAN vs Aadhaar): {'✓ PASS' if validation['name_match']['pass'] else '✗ FAIL — ' + validation['name_match']['detail']}
  DOB consistency:                   {'✓ PASS' if validation['dob_match']['pass'] else '✗ FAIL — CRITICAL MISMATCH'}
  PAN format validation:             {'✓ PASS' if validation['pan_format']['pass'] else '✗ FAIL'}
  PMLA mandatory fields:             {'✓ Complete' if validation['pmla_completeness']['pass'] else '✗ Missing: ' + str(validation['pmla_completeness']['missing'])}

Optional fields not captured: {validation.get('optional_missing', [])}

Note: This summary is based solely on extracted document data. No inferences made.
Onboarding officer should verify physical documents for final approval."""
    
    print(narrative)
    return narrative

# Run the full 3-stage pipeline
raw_ocr = """
INCOME TAX DEPARTMENT GOVT. OF INDIA
RAJESH KUMAR | Father: SURESH KUMAR | DOB: 15/03/1985
ABCDE1234F
---
UIDAI | Rajesh Kumar | 15/03/1985 | Male
XXXX XXXX 5678
42, MG Road, Bangalore - 560001
"""

print("=" * 65)
print("3-STAGE KYC PROMPT CHAIN EXECUTION")
print("=" * 65)

extracted = stage1_extract(raw_ocr)
validation = stage2_validate(extracted)
narrative = stage3_narrative(extracted, validation)

print("\n✓ Each stage is independently testable")
print("✓ Stage 1 can be swapped for a fine-tuned model without touching Stage 2 or 3")
print("✓ Each stage can have its own golden eval set and accuracy target")

## 4. RAG — Retrieval-Augmented Generation From Scratch

In [ ]:
# Build a minimal but complete RAG pipeline using only Python
# Uses TF-IDF for retrieval (replace with real embeddings when available)

class RAGPipeline:
    """
    A complete RAG pipeline:
    1. Chunk documents
    2. Index with TF-IDF (replace with neural embeddings in production)
    3. Retrieve relevant chunks for a query
    4. Inject into prompt and call LLM
    """
    
    def __init__(self):
        self.chunks = []
        self.tfidf_matrix = None
        self.vocab_idx = {}
        self.idf = None
    
    def _tokenize(self, text):
        stopwords = {'the','a','an','and','or','is','are','was','were','be','been',
                     'to','of','in','on','at','for','with','by','from','as','shall',
                     'will','all','per','that','this','which','it','its','not','any'}
        tokens = re.findall(r'\b[a-zA-Z][a-zA-Z0-9]*\b', text.lower())
        return [t for t in tokens if t not in stopwords and len(t) > 2]
    
    def chunk_document(self, text, chunk_size=150, overlap=30):
        """Simple word-count chunking with overlap"""
        words = text.split()
        chunks = []
        for i in range(0, len(words), chunk_size - overlap):
            chunk = ' '.join(words[i:i + chunk_size])
            if len(chunk.split()) > 20:  # Skip tiny tail chunks
                chunks.append(chunk)
        return chunks
    
    def add_documents(self, documents: dict):
        """documents: {doc_id: text}"""
        all_chunks = []
        for doc_id, text in documents.items():
            chunks = self.chunk_document(text)
            for i, chunk in enumerate(chunks):
                all_chunks.append({'doc_id': doc_id, 'chunk_id': i, 'text': chunk})
        
        self.chunks = all_chunks
        tokenized = [self._tokenize(c['text']) for c in all_chunks]
        
        vocab_set = set()
        for tokens in tokenized:
            vocab_set.update(tokens)
        vocab = sorted(vocab_set)
        self.vocab_idx = {w: i for i, w in enumerate(vocab)}
        
        from collections import Counter
        n = len(all_chunks)
        v = len(vocab)
        tf = np.zeros((n, v))
        for ci, tokens in enumerate(tokenized):
            counts = Counter(tokens)
            for word, cnt in counts.items():
                if word in self.vocab_idx:
                    tf[ci, self.vocab_idx[word]] = cnt / max(len(tokens), 1)
        
        df = (tf > 0).sum(axis=0)
        self.idf = np.log((n + 1) / (df + 1)) + 1
        self.tfidf_matrix = tf * self.idf
        norms = np.linalg.norm(self.tfidf_matrix, axis=1, keepdims=True)
        norms[norms == 0] = 1
        self.tfidf_matrix = self.tfidf_matrix / norms
        
        print(f"Indexed {n} chunks from {len(documents)} documents")
        return self
    
    def retrieve(self, query, top_k=3):
        tokens = self._tokenize(query)
        q_vec = np.zeros(len(self.vocab_idx))
        from collections import Counter
        counts = Counter(tokens)
        for word, cnt in counts.items():
            if word in self.vocab_idx:
                q_vec[self.vocab_idx[word]] = (cnt / max(len(tokens), 1)) * self.idf[self.vocab_idx[word]]
        norm = np.linalg.norm(q_vec)
        if norm > 0:
            q_vec = q_vec / norm
        
        scores = self.tfidf_matrix @ q_vec
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [(self.chunks[i], scores[i]) for i in top_idx if scores[i] > 0.01]
    
    def build_rag_prompt(self, query, top_k=3):
        """Build the full RAG prompt with retrieved context"""
        retrieved = self.retrieve(query, top_k=top_k)
        
        context_blocks = []
        for chunk, score in retrieved:
            context_blocks.append(
                f'<source doc="{chunk["doc_id"]}" relevance="{score:.3f}">\n{chunk["text"]}\n</source>'
            )
        
        context = "\n\n".join(context_blocks)
        
        system = """You are a banking policy Q&A assistant.
Answer ONLY based on the provided sources.
If the answer is not in the sources, say: "Not found in provided documents."
Always cite the source document name."""
        
        user = f"""<sources>
{context}
</sources>

Question: {query}"""
        
        return system, user, retrieved

print("RAGPipeline class defined ✓")

In [ ]:
# Banking policy knowledge base
POLICY_DOCUMENTS = {
    "RBI_KYC_2023": """
    RBI Master Direction on KYC 2023. All regulated entities shall carry out customer due diligence
    at the time of account opening. KYC documents include officially valid documents such as Aadhaar,
    Passport, Driving License, Voter ID, and PAN card. Video KYC is permitted as an alternative to
    physical KYC. Periodic KYC update is required every 2 years for high-risk customers, every 8 years
    for medium-risk, and every 10 years for low-risk customers. Customer identification includes
    name, address, date of birth, and identification number.
    """,
    
    "CREDIT_POLICY_2024": """
    Credit Underwriting Policy 2024. Maximum FOIR for salaried individuals is 55 percent.
    Maximum FOIR for self-employed individuals is 50 percent. Minimum CIBIL score is 700 for
    retail loans and 650 for MSME loans. Minimum employment tenure is 2 years for salaried.
    Loan to Value ratio for home loans: 80 percent for loans up to 30 lakhs, 75 percent for
    30-75 lakhs, and 70 percent above 75 lakhs. All home loans above 25 lakhs require independent
    property valuation by an empanelled valuer.
    """,
    
    "AML_POLICY_2024": """
    Anti-Money Laundering Policy 2024. Transaction monitoring system must flag: cash transactions
    above Rs 10 lakhs, structured transactions (multiple transactions just below reporting threshold),
    round-tripping of funds, and transactions inconsistent with customer profile. Suspicious
    Transaction Reports (STR) must be filed with FIU-IND within 7 days. Currency Transaction
    Reports (CTR) for cash above Rs 10 lakhs must be filed monthly. Enhanced Due Diligence is
    required for Politically Exposed Persons (PEPs) and high-risk jurisdictions.
    """,
    
    "INTEREST_RATE_POLICY": """
    Interest Rate Policy. Home loans are offered at floating rates linked to Repo Rate. Current
    rates: Salaried with CIBIL above 750 get Repo plus 2.0 percent. CIBIL 700-750 gets Repo plus
    2.5 percent. CIBIL below 700 is not eligible. Personal loans range from 10.5 to 18 percent
    depending on credit profile. Processing fee is 0.5 percent of loan amount subject to minimum
    of Rs 5000 and maximum of Rs 50000. Prepayment is free for floating rate loans.
    """
}

rag = RAGPipeline()
rag.add_documents(POLICY_DOCUMENTS)

print()

# Test queries
test_queries = [
    "What is the maximum debt to income ratio for self-employed borrowers?",
    "When does a customer need to update their KYC?",
    "What transactions should be reported to FIU-IND?",
    "What interest rate will a salaried customer with CIBIL 760 get?",
]

for query in test_queries[:2]:  # Show 2 in detail
    print(f"\n{'='*65}")
    print(f"QUERY: {query}")
    system, user, retrieved = rag.build_rag_prompt(query, top_k=2)
    print(f"\nRetrieved {len(retrieved)} relevant chunks:")
    for chunk, score in retrieved:
        print(f"  - [{chunk['doc_id']}] (score={score:.3f}): {chunk['text'][:100]}...")
    print(f"\nFull prompt sent to LLM:")
    print(f"SYSTEM: {system}")
    print(f"USER: {user[:400]}...")

## 5. Meta-Prompting — LLM Writes Its Own Prompts

In [ ]:
# Meta-prompting: using the LLM to generate and critique prompts
# Simulated responses showing what a real model would produce

meta_prompt_template = dedent("""
    You are an expert prompt engineer specializing in banking LLM applications.
    
    Task description: {task_description}
    
    Constraints:
    - Deployed in an RBI-regulated bank
    - Must never hallucinate missing data
    - Output must be machine-parseable JSON
    - Must handle edge cases gracefully
    
    Write 3 candidate system prompts.
    For each prompt, provide:
    1. The prompt text
    2. Design rationale (2 sentences)
    3. Potential failure mode
""").strip()

task = "Extract loan covenant details from legal documents and flag any breaches"

# Simulated meta-prompt output (what a real LLM would generate)
simulated_candidates = [
    {
        "version": "v1 - Conservative Extractor",
        "prompt": """You are a legal document analyst specializing in loan covenants.
Extract covenant details from the provided document.
For each covenant found, output: {"covenant_type": str, "threshold": str, "current_value": str|null, "breach_status": "Breach"|"Compliant"|"Unknown"}.
If a value is not explicitly stated, set it to null. Never infer or calculate values not present in the document.""",
        "rationale": "Strict null handling prevents hallucination of financial figures. Explicit 'Unknown' status acknowledges information gaps rather than guessing.",
        "failure_mode": "May mark too many fields as null if document uses non-standard terminology for common covenants."
    },
    {
        "version": "v2 - Reasoning Extractor",
        "prompt": """You are a credit analyst reviewing loan agreement covenants.
Step 1: List all covenants mentioned in the document.
Step 2: For each covenant, identify the threshold and any reported current value.
Step 3: Determine breach status based ONLY on explicitly stated values.
Output as JSON array. Mark confidence as High/Medium/Low per covenant.""",
        "rationale": "CoT structure ensures no covenants are missed and the reasoning is auditable. Confidence field enables downstream filtering.",
        "failure_mode": "Step-by-step format increases output token count by ~40%. At scale this adds cost."
    },
    {
        "version": "v3 - Strict Schema Enforcer",
        "prompt": """You are a covenant extraction engine. Respond ONLY with JSON matching this schema:
{"covenants": [{"name": str, "type": "Financial|Operational|Reporting|Other", "threshold": str|null,
"current_value": str|null, "breach": boolean|null, "source_clause": str}], "extraction_complete": boolean}
Set 'breach' to null when insufficient data. Set 'extraction_complete' to false if document appears truncated.""",
        "rationale": "Rigid schema prevents format variation between runs. source_clause provides audit trail linking each extracted value to document text.",
        "failure_mode": "Strict JSON constraint may cause failures on very long covenant lists that exceed context window mid-output."
    }
]

print(f"META-PROMPT INPUT: {task}\n")
print("=" * 65)
print("GENERATED CANDIDATE PROMPTS:")
print("=" * 65)

for c in simulated_candidates:
    print(f"\n--- {c['version']} ---")
    print(f"Prompt:\n{c['prompt']}")
    print(f"\nRationale: {c['rationale']}")
    print(f"Failure mode: {c['failure_mode']}")

print("\n" + "="*65)
print("NEXT STEP: Run all 3 on your golden eval set. Pick the winner.")
print("NEVER deploy meta-generated prompts without eval set validation.")

## 6. Prompt Compression — Fitting Long Documents

In [ ]:
# Demonstrate prompt compression strategies

class PromptCompressor:
    """Simple rule-based compression (LLMLingua uses a small LM for this)"""
    
    # Common filler patterns that add tokens without adding information
    FILLER_PATTERNS = [
        (r'\bIt is (important|necessary|essential|crucial) to note that\b', ''),
        (r'\bAs (mentioned|stated|described) (above|below|earlier|previously)\b', ''),
        (r'\bIn accordance with (the provisions of )?\b', 'Per '),
        (r'\bFor the (purposes|purpose) of (this|the) \b', 'For '),
        (r'\bWith (respect|regard|reference) to\b', 'Re:'),
        (r'\b(shall|will) be (required|obligated|mandated) to\b', 'must'),
        (r'\bnotwithstanding the (foregoing|above|provisions of this clause)\b', 'regardless'),
        (r'\bin the event that\b', 'if'),
        (r'\bprior to\b', 'before'),
        (r'\bsubsequent to\b', 'after'),
        (r'\bfor the avoidance of doubt\b', ''),
        (r'\bhereinafter referred to as\b', '='),
    ]
    
    def compress(self, text):
        compressed = text
        for pattern, replacement in self.FILLER_PATTERNS:
            compressed = re.sub(pattern, replacement, compressed, flags=re.IGNORECASE)
        # Collapse multiple spaces
        compressed = re.sub(r'  +', ' ', compressed)
        # Remove empty lines
        compressed = re.sub(r'\n\s*\n\s*\n', '\n\n', compressed)
        return compressed.strip()
    
    def estimate_tokens(self, text):
        return int(len(text.split()) * 1.3)
    
    def compression_report(self, original, compressed):
        orig_tokens = self.estimate_tokens(original)
        comp_tokens = self.estimate_tokens(compressed)
        ratio = comp_tokens / orig_tokens
        return {
            'original_tokens': orig_tokens,
            'compressed_tokens': comp_tokens,
            'compression_ratio': round(ratio, 3),
            'token_savings': orig_tokens - comp_tokens,
            'savings_pct': round((1 - ratio) * 100, 1)
        }

# Legal text typical of banking documents
legal_text = """
It is important to note that in accordance with the provisions of the Master Direction on 
Know Your Customer (KYC), as mentioned above, all Scheduled Commercial Banks shall be required 
to carry out customer due diligence prior to the establishment of any business relationship. 
Notwithstanding the foregoing, in the event that the customer is an existing customer who has 
previously submitted KYC documentation, the bank may, with respect to such customer, accept 
a self-declaration form in lieu of fresh documentation, subsequent to the completion of the 
requisite verification process. For the avoidance of doubt, this provision shall not apply 
to high-risk customers as defined hereinafter referred to as HRC, for whom enhanced due 
diligence will be required in accordance with the provisions of Chapter V of the said 
Master Direction.
""".strip()

compressor = PromptCompressor()
compressed = compressor.compress(legal_text)
report = compressor.compression_report(legal_text, compressed)

print("ORIGINAL TEXT:")
print(legal_text)
print("\nCOMPRESSED TEXT:")
print(compressed)
print("\nCOMPRESSION REPORT:")
for k, v in report.items():
    print(f"  {k}: {v}")

print("\n--- Production-grade LLMLingua goes further ---")
print("It uses a small LM to identify which tokens are low-information")
print("and removes them, achieving 3-5x compression with <5% accuracy loss.")
print("Key use case: fitting a 100-page RBI circular into a 128K context window.")

## Summary: Advanced Patterns Cheat Sheet

| Pattern | When to Use | Cost | Complexity |
|---------|------------|------|------------|
| **ReAct** | Multi-step decisions requiring real-time data | Medium (multiple calls) | High |
| **Self-consistency** | High-stakes binary decisions, uncertainty detection | High (N× calls) | Low |
| **Prompt chaining** | Any multi-step workflow | Medium | Medium |
| **RAG** | Q&A over private documents, policy lookup | Low (retrieval cheap) | High |
| **Meta-prompting** | Prompt development phase only | Low (dev time) | Low |
| **Compression** | Long documents, context window limits | Negligible | Low |

**Decision tree:**
1. Start simple → Zero-shot
2. Need examples → Few-shot  
3. Need reasoning → CoT
4. Need external data → ReAct
5. Need reliability on high-stakes → Self-consistency
6. Need domain knowledge → RAG
7. Complex multi-step → Prompt chain